# 11_baseline_growth_history_260513

Conservative baseline growth history modeling notebook.


In [1]:
# Step 11 conservative baseline growth history. Executed inside this notebook.
from pathlib import Path
from datetime import datetime
import subprocess, warnings, zipfile, json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

STEP = "11_baseline_growth_history_260513"
EXPECTED_ROOTS = {"C:/Code/ott-churn-prediction", "C:\\Code\\ott-churn-prediction"}
actual_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
print("repo root:", actual_root)
if actual_root not in EXPECTED_ROOTS:
    raise SystemExit(f"STOP: repo root mismatch: {actual_root}")

ROOT = Path(actual_root)
PARK = ROOT / "park.ingyeom"
NOTEBOOK = PARK / "notebook" / STEP / f"{STEP}.ipynb"
NOTE = PARK / "note.md"
BASE_MODEL = PARK / "reports" / "models" / STEP
BASE_FIG = PARK / "reports" / "figures" / STEP
ZIP_DIR = PARK / "zip"
ZIP_PATH = ZIP_DIR / f"{STEP}_review_package.zip"

def inside_park(p):
    try:
        Path(p).resolve().relative_to(PARK.resolve())
        return True
    except Exception:
        return False

for p in [NOTEBOOK, NOTE, BASE_MODEL, BASE_FIG, ZIP_DIR]:
    assert inside_park(p), f"outside park.ingyeom blocked: {p}"

def choose_dir(base):
    base.mkdir(parents=True, exist_ok=True)
    if any(base.iterdir()):
        d = base / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        d.mkdir(parents=True, exist_ok=False)
        return d
    return base

MODEL_DIR = choose_dir(BASE_MODEL)
FIG_DIR = choose_dir(BASE_FIG)
print("actual model output folder:", MODEL_DIR)
print("actual figure output folder:", FIG_DIR)

P = {
    "primary": PARK/"reports/audits/06_common_preprocessing_and_final_cohort_260513/06_primary_main_cohort_conservative_features.csv",
    "index": PARK/"reports/audits/06_common_preprocessing_and_final_cohort_260513/06_primary_main_cohort_index.csv",
    "canon": PARK/"reports/audits/05b_column_role_dictionary_patch_260513/05b_canonical_column_role_dictionary.csv",
    "safe": PARK/"reports/audits/05b_column_role_dictionary_patch_260513/05b_conservative_safe_candidate_columns.csv",
    "review": PARK/"reports/audits/05b_column_role_dictionary_patch_260513/05b_review_required_columns.csv",
    "forbid": PARK/"reports/audits/05b_column_role_dictionary_patch_260513/05b_forbidden_drop_columns.csv",
    "aarr": PARK/"reports/audits/07_AARRR_feature_mapping_260513/07_AARRR_mapping_conservative_features.csv",
    "handoff": PARK/"reports/audits/07_AARRR_feature_mapping_260513/07_AARRR_to_baseline_ladder_handoff.csv",
    "09_final": PARK/"reports/eda/09_promotion_repurchase_2x2_eda_260513/09_final_checks.csv",
    "09_top": PARK/"reports/eda/09_promotion_repurchase_2x2_eda_260513/09_top_target_signals_by_group.csv",
    "09_cross": PARK/"reports/eda/09_promotion_repurchase_2x2_eda_260513/09_cross_group_target_signal_comparison.csv",
    "09_contrast": PARK/"reports/eda/09_promotion_repurchase_2x2_eda_260513/09_08_vs_09_contrast_summary.csv",
}
P09B = PARK/"reports/audits/09b_raw_view_window_validation_260514/run_20260514_130402"
R09B = ["09b_final_checks.csv","09b_core_usage_recalculation_comparison.csv","09b_day21_plus_leakage_contrast_test.csv","09b_window_validation_decision.csv"]
R10 = ["10_final_checks.csv","10_feature_eda_catalog.csv","10_focus_feature_deep_dive_summary.csv","10_handoff_to_11_and_17.csv","10_open_risks_for_next_steps.csv"]

def all_pass(fp):
    if not Path(fp).exists():
        return False
    x = pd.read_csv(fp)
    return "status" in x.columns and x["status"].fillna("").eq("PASS").all()

def detect_10():
    base = PARK/"reports/eda/10_feature_eda_260513"
    if all((base/n).exists() for n in R10) and all_pass(base/"10_final_checks.csv"):
        return base
    runs = []
    if base.exists():
        for d in base.glob("run_*"):
            if d.is_dir() and all((d/n).exists() for n in R10) and all_pass(d/"10_final_checks.csv"):
                runs.append(d)
    return sorted(runs)[-1] if runs else None

P10 = detect_10()
required = list(P.values()) + [P09B/n for n in R09B] + ([] if P10 is None else [P10/n for n in R10])
missing = [str(p) for p in required if not p.exists()]
pre = []
def pre_row(name, ok, value="", notes=""):
    pre.append(dict(check_name=name,status="PASS" if ok else "FAIL",value=value,notes=notes,
                    expected_repo_root="C:/Code/ott-churn-prediction",actual_repo_root=actual_root,
                    detected_09b_output_folder=str(P09B),detected_10_output_folder=str(P10 or ""),
                    actual_model_output_folder=str(MODEL_DIR),actual_figure_output_folder=str(FIG_DIR)))
pre_row("expected_repo_root", True, "C:/Code/ott-churn-prediction")
pre_row("actual_repo_root", True, actual_root)
pre_row("repo_root_match", actual_root in EXPECTED_ROOTS)
pre_row("all_required_input_files_exist", not missing, len(required), ";".join(missing[:20]))
pre_row("detected_09b_output_folder", P09B.exists(), str(P09B))
pre_row("detected_10_output_folder", P10 is not None, str(P10 or ""))
pre_row("primary_table_exists", P["primary"].exists())
pre_row("note_md_exists", NOTE.exists(), notes="will create if missing")
pre_row("model_output_folder_inside_park_ingyeom", inside_park(MODEL_DIR), str(MODEL_DIR))
pre_row("figure_output_folder_inside_park_ingyeom", inside_park(FIG_DIR), str(FIG_DIR))
pre_row("notebook_inside_park_ingyeom", inside_park(NOTEBOOK), str(NOTEBOOK))
pre_row("zip_folder_inside_park_ingyeom", inside_park(ZIP_DIR), str(ZIP_DIR))
pre_row("09b_final_checks_all_pass", all_pass(P09B/"09b_final_checks.csv"))
pre_row("10_final_checks_all_pass", P10 is not None and all_pass(P10/"10_final_checks.csv"))
can = (actual_root in EXPECTED_ROOTS) and not missing and P09B.exists() and all_pass(P09B/"09b_final_checks.csv") and P10 is not None
pre_row("can_proceed", can, can)
pd.DataFrame(pre).to_csv(MODEL_DIR/"11_preflight_input_validation.csv", index=False, encoding="utf-8-sig")
if not can:
    (MODEL_DIR/"README.md").write_text(f"# {STEP}\n\nPreflight failed. See `11_preflight_input_validation.csv`.\n", encoding="utf-8")
    raise SystemExit("STOP: preflight failed")

df = pd.read_csv(P["primary"])
safe = pd.read_csv(P["safe"])
review = pd.read_csv(P["review"])
forbid = pd.read_csv(P["forbid"])
aarr = pd.read_csv(P["aarr"])
handoff = pd.read_csv(P["handoff"])
top09 = pd.read_csv(P["09_top"])
cat10 = pd.read_csv(P10/"10_feature_eda_catalog.csv")
focus10 = pd.read_csv(P10/"10_focus_feature_deep_dive_summary.csv")

TARGET, SPLIT, GROUP = "is_repurchase", "is_promotion", "USER_KEY"
BLOCK = {GROUP, "source_row_number", TARGET, "repurchase_score", "churn_risk"}
review_cols = set(review["column_name"].dropna().astype(str))
forbid_cols = set(forbid["column_name"].dropna().astype(str))
safe_all = [c for c in safe["column_name"].dropna().astype(str).tolist() if c in df.columns and c not in BLOCK and c != SPLIT]

warns = []
def warn(tp, scope="", ladder="", model="", fold="", severity="WARNING", msg=""):
    warns.append(dict(warning_type=tp,dataset_scope=scope,ladder_step=ladder,model_name=model,fold=fold,
                      severity=severity,message=msg,actual_model_output_folder=str(MODEL_DIR),actual_figure_output_folder=str(FIG_DIR)))

def handoff_cols(label):
    r = handoff[handoff["proposed_name"].astype(str).str.contains(label, regex=False, na=False)]
    if r.empty: return []
    val = str(r.iloc[0].get("candidate_columns_from_conservative_table",""))
    return [c for c in val.split(";") if c in safe_all and c in df.columns]

L1 = handoff_cols("L1_activation_safe_window")
L2add = handoff_cols("L2_retention_w2_safe_window")
L3add = handoff_cols("L3_retention_w3_safe_window")
L2 = list(dict.fromkeys(L1 + L2add))
L3 = list(dict.fromkeys(L2 + L3add))
L4 = list(dict.fromkeys(safe_all))
orders = {"L0_dummy_prior":0,"L1_activation_safe":1,"L2_add_week2_retention":2,"L3_add_week3_retention":3,"L4_all_conservative_behavior":4,"L5_all_conservative_plus_promotion_indicator":5}
meta = {str(r["column_name"]):dict(AARRR_stage=r.get("AARRR_stage_primary",""),feature_family=r.get("feature_family","")) for _,r in aarr.iterrows()}

def numeric_only(cols, scope, ladder):
    out, bad = [], []
    for c in cols:
        s = pd.to_numeric(df[c], errors="coerce")
        if df[c].notna().sum() and s.notna().sum() == 0: bad.append(c)
        else: out.append(c)
    if bad: warn("non_numeric_feature_excluded", scope, ladder, msg=";".join(bad))
    return out

scopes = {
    "overall_without_promotion": df.index.to_numpy(),
    "overall_with_promotion": df.index.to_numpy(),
    "promotion_only": df.index[df[SPLIT] == 1].to_numpy(),
    "nonpromotion_only": df.index[df[SPLIT] == 0].to_numpy(),
}
scope_rows = []
for s, idx in scopes.items():
    d = df.loc[idx]
    n = len(d); pos = int((d[TARGET]==1).sum())
    scope_rows.append(dict(dataset_scope=s,row_filter="all rows" if s.startswith("overall") else ("is_promotion == 1" if s=="promotion_only" else "is_promotion == 0"),
        row_count=n,repurchase_count=pos,nonrepurchase_count=n-pos,repurchase_rate=pos/n if n else np.nan,
        unique_USER_KEY_count=d[GROUP].nunique(),duplicated_USER_KEY_extra_rows=n-d[GROUP].nunique(),
        is_promotion_feature_allowed=s=="overall_with_promotion",is_promotion_feature_used="only L5" if s=="overall_with_promotion" else "no",
        reason="conservative baseline scope",caution="Rows are subscription-event-level, not unique users; no causal claim."))
pd.DataFrame(scope_rows).to_csv(MODEL_DIR/"11_dataset_scope_definition.csv", index=False, encoding="utf-8-sig")

ladder_rows = []
base = {"L0_dummy_prior":[],"L1_activation_safe":L1,"L2_add_week2_retention":L2,"L3_add_week3_retention":L3,"L4_all_conservative_behavior":L4}
for scope in scopes:
    defs = dict(base)
    if scope == "overall_with_promotion":
        defs["L5_all_conservative_plus_promotion_indicator"] = L4 + [SPLIT]
    for lad, cols in defs.items():
        feats = [] if lad=="L0_dummy_prior" else list(dict.fromkeys(cols))
        feats = [c for c in feats if c not in BLOCK and not (c == SPLIT and not (scope=="overall_with_promotion" and lad.startswith("L5")))]
        feats = numeric_only(feats, scope, lad) if feats else []
        if lad!="L0_dummy_prior" and not feats: warn("empty_feature_step", scope, lad, msg="zero usable feature columns")
        added = feats if lad in ["L1_activation_safe","L4_all_conservative_behavior","L5_all_conservative_plus_promotion_indicator"] else [c for c in feats if c not in (L1 if lad.startswith("L2") else L2)]
        stages = sorted({meta.get(c,{}).get("AARRR_stage","") for c in feats if c in meta})
        ladder_rows.append(dict(dataset_scope=scope,ladder_step=lad,ladder_order=orders[lad],feature_names=";".join(feats),
            feature_count=0 if lad=="L0_dummy_prior" else len(feats),added_features=";".join(added),AARRR_stage=";".join([x for x in stages if x]),
            reason="dummy prior floor baseline" if lad=="L0_dummy_prior" else "conservative safe-window behavioral ladder",
            allowed_status="allowed" if feats or lad=="L0_dummy_prior" else "warning_empty",
            warning_if_empty="" if feats or lad=="L0_dummy_prior" else "WARNING: zero usable feature columns",
            review_columns_excluded_count=len([c for c in review_cols if c in df.columns and c not in feats]),
            forbidden_columns_excluded_count=len([c for c in forbid_cols if c in df.columns and c not in feats])))
ladder_df = pd.DataFrame(ladder_rows)
ladder_df.to_csv(MODEL_DIR/"11_feature_ladder_definition.csv", index=False, encoding="utf-8-sig")

pd.DataFrame([dict(input_table_path=str(P["primary"]),row_count=len(df),column_count=df.shape[1],target_column=TARGET,split_column=SPLIT,group_key=GROUP,
    conservative_feature_count=len(safe_all),review_column_count=len(review_cols),forbidden_column_count=len(forbid_cols),
    target_distribution=json.dumps(df[TARGET].value_counts().to_dict(), ensure_ascii=False),
    promotion_distribution=json.dumps(df[SPLIT].value_counts().to_dict(), ensure_ascii=False),
    no_review_columns_used=True,no_forbidden_columns_used=True,no_source_row_number_as_feature=True,no_USER_KEY_as_feature=True,no_is_repurchase_as_feature=True,
    no_repurchas_score_or_churn_risk_as_input=True,**{"09b_window_validation_status":"PASS","10_feature_eda_status":"PASS"},
    interpretation="Conservative safe-window behavioral baseline ladder; not final model, not causal.")]).to_csv(MODEL_DIR/"11_modeling_input_contract.csv", index=False, encoding="utf-8-sig")

registry = [
["DummyPrior","sklearn.dummy.DummyClassifier","L0_dummy_prior","none; formal X only","strategy=prior","no","floor baseline","constant predictions","feature_count recorded as 0"],
["LogisticRegression","sklearn.linear_model.LogisticRegression","L1-L5 where allowed","SimpleImputer(median) + StandardScaler","max_iter=2000, solver=lbfgs","no","stable linear baseline","underfit possible","numeric columns only"],
["HistGradientBoosting","sklearn.ensemble.HistGradientBoostingClassifier","L1-L5 where allowed","SimpleImputer(median)","random_state=42","no","stable sklearn tree baseline","train-valid gap possible","not tuned; high AUC with gap is not automatically safe"],
["RandomForest","sklearn.ensemble.RandomForestClassifier","L1-L5 where allowed","SimpleImputer(median)","n_estimators=300, max_depth=6, min_samples_leaf=20, n_jobs=-1, random_state=42","no","baseline comparison only","overfit gap possible","not final model"]]
reg_df = pd.DataFrame(registry, columns=["model_name","sklearn_class","intended_ladder_steps","preprocessing","fixed_parameters","tuning_performed","reason","expected_risk","caution"])
reg_df.to_csv(MODEL_DIR/"11_model_registry.csv", index=False, encoding="utf-8-sig")
models = {
    "DummyPrior": DummyClassifier(strategy="prior"),
    "LogisticRegression": Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("model",LogisticRegression(max_iter=2000,solver="lbfgs"))]),
    "HistGradientBoosting": Pipeline([("imputer",SimpleImputer(strategy="median")),("model",HistGradientBoostingClassifier(random_state=42))]),
    "RandomForest": Pipeline([("imputer",SimpleImputer(strategy="median")),("model",RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42))])
}

cv_rows, metric_rows, oof = [], [], {}
splits = {}
for scope, idx in scopes.items():
    d = df.loc[idx].reset_index(drop=False).rename(columns={"index":"orig_index"})
    y = d[TARGET].astype(int).to_numpy(); g = d[GROUP].to_numpy()
    try:
        sp = list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42).split(np.zeros(len(d)), y, groups=g))
    except Exception as e:
        warn("cv_scope_failed",scope,severity="FAIL",msg=str(e)); splits[scope]=None; continue
    ok = []
    for fold,(tr,va) in enumerate(sp,1):
        ov = len(set(g[tr]).intersection(set(g[va]))); both = len(np.unique(y[va]))==2
        status = "PASS" if ov==0 and both else "FAIL"
        if status!="PASS": warn("cv_fold_failed",scope,fold=fold,severity="FAIL",msg=f"overlap={ov};both_classes={both}")
        cv_rows.append(dict(dataset_scope=scope,fold=fold,train_row_count=len(tr),valid_row_count=len(va),train_repurchase_rate=y[tr].mean(),
            valid_repurchase_rate=y[va].mean(),train_unique_USER_KEY_count=len(set(g[tr])),valid_unique_USER_KEY_count=len(set(g[va])),
            group_overlap_count_between_train_valid=ov,valid_promotion_rate=d.loc[va,SPLIT].mean(),fold_valid_class_has_both_classes=both,status=status,
            warning="" if status=="PASS" else "CV fold failed group/class audit"))
        if status=="PASS": ok.append((tr,va))
    splits[scope] = (d, ok) if len(ok)==5 else None
    if len(ok)!=5: warn("cv_scope_failed",scope,severity="FAIL",msg="not all five folds passed")
pd.DataFrame(cv_rows).to_csv(MODEL_DIR/"11_cv_split_audit.csv", index=False, encoding="utf-8-sig")

for _, r in ladder_df.sort_values(["dataset_scope","ladder_order"]).iterrows():
    scope, lad = r["dataset_scope"], r["ladder_step"]
    feats = [] if lad=="L0_dummy_prior" else [c for c in str(r["feature_names"]).split(";") if c]
    mnames = ["DummyPrior"] if lad=="L0_dummy_prior" else ["LogisticRegression","HistGradientBoosting","RandomForest"]
    if splits.get(scope) is None:
        for mn in mnames: warn("experiment_skipped_cv_unavailable",scope,lad,mn,severity="SKIP",msg="5-fold StratifiedGroupKFold unavailable")
        continue
    if lad!="L0_dummy_prior" and not feats:
        for mn in mnames: warn("experiment_skipped_empty_features",scope,lad,mn,severity="SKIP",msg="zero usable feature columns")
        continue
    d, sp = splits[scope]; y = d[TARGET].astype(int).to_numpy(); orig = d["orig_index"].to_numpy()
    X = np.zeros((len(d),1)) if lad=="L0_dummy_prior" else d[feats].apply(pd.to_numeric, errors="coerce")
    for mn in mnames:
        pred = np.full(len(d), np.nan); fold_id = np.full(len(d), np.nan); trains=[]; vals=[]; aps=[]; brs=[]
        for fold,(tr,va) in enumerate(sp,1):
            mdl = clone(models[mn])
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    mdl.fit(X.iloc[tr] if hasattr(X,"iloc") else X[tr], y[tr])
                    p_tr = mdl.predict_proba(X.iloc[tr] if hasattr(X,"iloc") else X[tr])[:,1]
                    p_va = mdl.predict_proba(X.iloc[va] if hasattr(X,"iloc") else X[va])[:,1]
                for w in caught: warn("sklearn_fit_warning",scope,lad,mn,fold,msg=str(w.message))
                ta = roc_auc_score(y[tr], p_tr); vaa = roc_auc_score(y[va], p_va); ap = average_precision_score(y[va], p_va); br = brier_score_loss(y[va], p_va)
                pred[va]=p_va; fold_id[va]=fold; trains.append(ta); vals.append(vaa); aps.append(ap); brs.append(br)
                metric_rows.append(dict(dataset_scope=scope,ladder_step=lad,model_name=mn,fold=fold,train_auc=ta,valid_auc=vaa,train_valid_gap=ta-vaa,
                    valid_average_precision=ap,valid_brier_score=br,train_row_count=len(tr),valid_row_count=len(va),
                    feature_count=0 if lad=="L0_dummy_prior" else len(feats),warning="; ".join(str(w.message) for w in caught)))
            except Exception as e:
                warn("model_fit_failed",scope,lad,mn,fold,severity="FAIL",msg=str(e))
        if np.isfinite(pred).all():
            oof[(scope,lad,mn)] = dict(orig_index=orig,fold=fold_id,pred=pred,y=y,feature_count=0 if lad=="L0_dummy_prior" else len(feats),trains=trains,vals=vals,aps=aps,brs=brs)
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(MODEL_DIR/"11_cv_fold_metrics.csv", index=False, encoding="utf-8-sig")

summary = []
for (scope,lad,mn), rec in oof.items():
    tr = np.array(rec["trains"]); va = np.array(rec["vals"]); gap = tr-va
    oauc = roc_auc_score(rec["y"], rec["pred"]); oap = average_precision_score(rec["y"], rec["pred"]); obr = brier_score_loss(rec["y"], rec["pred"])
    mg = float(np.nanmean(gap)); sd = float(np.nanstd(va,ddof=1)) if len(va)>1 else 0.0
    if mg >= .12: st, caut = "overfit_warning", "Large train-valid AUC gap; high AUC is not automatically safe."; warn("overfit_warning",scope,lad,mn,msg=caut)
    elif sd >= .06: st, caut = "unstable", "Fold AUC variability is high."; warn("unstable_fold_warning",scope,lad,mn,msg=caut)
    elif oauc < .55: st, caut = "weak", "Weak baseline signal."
    else: st, caut = "usable_baseline", "Baseline only; not final model."
    summary.append(dict(dataset_scope=scope,ladder_step=lad,model_name=mn,feature_count=rec["feature_count"],n_folds_completed=len(va),
        mean_valid_auc=float(np.nanmean(va)),std_valid_auc=sd,min_valid_auc=float(np.nanmin(va)),max_valid_auc=float(np.nanmax(va)),
        mean_train_auc=float(np.nanmean(tr)),mean_train_valid_gap=mg,mean_valid_average_precision=float(np.nanmean(rec["aps"])),
        mean_valid_brier_score=float(np.nanmean(rec["brs"])),oof_auc=float(oauc),oof_average_precision=float(oap),oof_brier_score=float(obr),
        performance_status=st,caution=caut))
summ = pd.DataFrame(summary)
summ.to_csv(MODEL_DIR/"11_cv_summary_metrics.csv", index=False, encoding="utf-8-sig")

growth = []
for (scope,mn), s in summ.groupby(["dataset_scope","model_name"]):
    s = s.assign(order=s["ladder_step"].map(orders)).sort_values("order")
    prev = np.nan
    l0 = summ[(summ["dataset_scope"]==scope) & (summ["ladder_step"]=="L0_dummy_prior")]["oof_auc"]
    l0v = float(l0.iloc[0]) if len(l0) else np.nan
    for _, r in s.iterrows():
        cur = float(r["oof_auc"])
        growth.append(dict(dataset_scope=scope,model_name=mn,ladder_step=r["ladder_step"],mean_valid_auc=r["mean_valid_auc"],oof_auc=cur,
            delta_oof_auc_from_previous_step=np.nan if np.isnan(prev) else cur-prev,delta_oof_auc_from_L0=np.nan if np.isnan(l0v) else cur-l0v,
            added_feature_family="dummy prior" if r["ladder_step"]=="L0_dummy_prior" else ("promotion indicator" if r["ladder_step"].startswith("L5") else "conservative safe-window behavior"),
            interpretation="Prediction separation change across conservative feature ladder; not causal.",caution="No threshold, no segmentation, no deployment readiness claim."))
        prev = cur
pd.DataFrame(growth).to_csv(MODEL_DIR/"11_ladder_growth_summary.csv", index=False, encoding="utf-8-sig")

best_rows, selected = [], set()
for scope, s in summ.groupby("dataset_scope"):
    high = s.sort_values(["oof_auc","mean_valid_auc"], ascending=False).iloc[0]
    pool = s[(s["performance_status"]=="usable_baseline") & (s["mean_train_valid_gap"]<=.08)]
    safer = high if pool.empty else pool.sort_values(["oof_auc","mean_valid_auc"], ascending=False).iloc[0]
    selected.add((scope, str(safer["ladder_step"]), str(safer["model_name"])))
    best_rows.append(dict(dataset_scope=scope,best_model_name=safer["model_name"],best_ladder_step=safer["ladder_step"],best_oof_auc=safer["oof_auc"],
        mean_valid_auc=safer["mean_valid_auc"],train_valid_gap=safer["mean_train_valid_gap"],feature_count=safer["feature_count"],
        highest_auc_candidate=f"{high['model_name']} / {high['ladder_step']} / oof_auc={high['oof_auc']:.6f}",
        safer_followup_candidate=f"{safer['model_name']} / {safer['ladder_step']} / oof_auc={safer['oof_auc']:.6f}",
        why_selected="Selected among usable baselines with low train-valid gap." if not pool.empty else "No clearly safer low-gap usable candidate; use highest AUC with caution.",
        why_not_final_model="Step 11 baseline only; no SHAP, tuning, threshold, or segmentation.",
        caution="AUC winner and presentation-safe follow-up candidate are intentionally distinguished."))
best = pd.DataFrame(best_rows)
best.to_csv(MODEL_DIR/"11_best_baseline_by_scope.csv", index=False, encoding="utf-8-sig")
cons = summ[~summ["ladder_step"].str.contains("L5",na=False)]
if len(cons):
    r = cons.sort_values(["oof_auc","mean_valid_auc"], ascending=False).iloc[0]
    selected.add((str(r["dataset_scope"]),str(r["ladder_step"]),str(r["model_name"])))

def bucket(g):
    if pd.isna(g): return "unknown"
    return "low" if g<.03 else ("moderate" if g<.08 else ("high" if g<.12 else "severe"))
gap_rows = [dict(dataset_scope=r.dataset_scope,ladder_step=r.ladder_step,model_name=r.model_name,mean_train_auc=r.mean_train_auc,mean_valid_auc=r.mean_valid_auc,
    gap=r.mean_train_valid_gap,gap_bucket=bucket(r.mean_train_valid_gap),overfit_warning=bucket(r.mean_train_valid_gap) in ["high","severe"],
    interpretation="High gap weakens safety even when AUC is high." if bucket(r.mean_train_valid_gap) in ["high","severe"] else "No major gap warning for baseline audit.")
    for r in summ.itertuples()]
pd.DataFrame(gap_rows).to_csv(MODEL_DIR/"11_train_valid_gap_audit.csv", index=False, encoding="utf-8-sig")

oof_rows, man = [], []
for key in sorted(selected):
    if key not in oof: continue
    scope, lad, mn = key; rec = oof[key]; dsel = df.loc[rec["orig_index"]].reset_index(drop=True)
    for i in range(len(dsel)):
        oof_rows.append(dict(source_row_number=dsel.loc[i].get("source_row_number",""),USER_KEY=dsel.loc[i,GROUP],is_promotion=dsel.loc[i,SPLIT],is_repurchase=dsel.loc[i,TARGET],
            dataset_scope=scope,selected_model_name=mn,selected_ladder_step=lad,fold=int(rec["fold"][i]),repurchase_score=float(rec["pred"][i]),
            churn_risk=float(1-rec["pred"][i]),note="Score orientation audit only; not segmentation or targeting threshold."))
    man.append(dict(selected_experiment=f"{scope}::{lad}::{mn}",reason_selected="best baseline per scope or overall best conservative baseline",row_count=len(dsel),
        score_orientation="repurchase_score = P(is_repurchase=1); churn_risk = 1 - repurchase_score",allowed_use="audit score orientation and baseline review only",
        forbidden_use="no segmentation candidate, no targeting criterion, no final threshold"))
pd.DataFrame(oof_rows).drop_duplicates().to_csv(MODEL_DIR/"11_oof_predictions_selected_baselines.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(man).to_csv(MODEL_DIR/"11_oof_prediction_manifest.csv", index=False, encoding="utf-8-sig")

best_feats = set()
for _, r in best.iterrows():
    x = ladder_df[(ladder_df.dataset_scope==r.dataset_scope)&(ladder_df.ladder_step==r.best_ladder_step)]
    if len(x): best_feats.update([c for c in str(x.iloc[0].feature_names).split(";") if c])
use = []
topset = set(top09["feature_name"].astype(str)) if "feature_name" in top09.columns else set()
focusset = set(focus10["feature_name"].astype(str)) if "feature_name" in focus10.columns else set()
for f in L4:
    steps = sorted(ladder_df[ladder_df.feature_names.fillna("").str.contains(f,regex=False)].ladder_step.unique(), key=lambda x: orders.get(x,99))
    use.append(dict(feature_name=f,used_in_ladder_steps=";".join(steps),AARRR_stage=meta.get(f,{}).get("AARRR_stage",""),
        feature_family=meta.get(f,{}).get("feature_family",""),appears_in_09_top_signal=f in topset,appears_in_10_focus_feature=f in focusset,
        included_in_best_baseline_any_scope=f in best_feats,caution="Conservative behavior feature; descriptive model input only."))
pd.DataFrame(use).to_csv(MODEL_DIR/"11_feature_usage_by_ladder.csv", index=False, encoding="utf-8-sig")

pd.DataFrame([dict(policy_item="score_orientation",target_positive_class="is_repurchase=1",repurchase_score="predicted probability for repurchase",
    churn_risk="1 - repurchase_score",auc_orientation="AUC computed with repurchase as positive class",high_repurchase_score_means="higher predicted repurchase probability and lower churn risk",
    high_churn_risk_means="higher non-repurchase risk",threshold_policy="No final threshold is chosen in this step")]).to_csv(MODEL_DIR/"11_score_orientation_policy.csv", index=False, encoding="utf-8-sig")

pd.DataFrame([
["이 모델이 이탈 원인을 밝혔다.","이 모델은 보수적 1~3주차 행동 feature를 바탕으로 재구매 가능성을 예측하는 baseline이다."],
["AUC가 높으니 마케팅 효과가 입증됐다.","AUC는 재구매 여부를 구분하는 예측 성능 지표이며, 마케팅 효과나 인과효과를 의미하지 않는다."],
["churn_risk 상위 고객을 바로 타겟팅하면 된다.","이 단계의 churn_risk는 baseline audit score이며, 운영 타겟팅 기준은 세그먼트와 실험 설계 이후에 정해야 한다."],
["review 컬럼도 넣으면 성능이 오르니 쓰자.","review 컬럼은 timing/semantic 확인 전까지 표준 baseline에 넣지 않는다. 필요하면 sensitivity 실험으로 분리한다."],
["프로모션 전용 모델에 is_promotion을 넣었다.","프로모션 전용 모델에서는 is_promotion이 상수이므로 feature에서 제외한다."]], columns=["unsafe_wording","safer_wording"]).to_csv(MODEL_DIR/"11_safe_unsafe_wording.csv", index=False, encoding="utf-8-sig")

risks = ["Step 11 is baseline modeling only, not final model.","No SHAP yet.","No Optuna yet.","No final threshold yet.","No segmentation yet.","Review columns remain excluded.","Membership/context L0 remains limited under conservative approach.","AUC may be modest because feature set is conservative.","Group-aware CV must remain in future modeling.","12 should compare broader model families carefully.","16 SHAP later must use final selected model, not every exploratory model.","17 segmentation later must not use 11 bins or scores as final thresholds without design.","Selected OOF scores are score orientation audit outputs only, not targeting criteria."]
pd.DataFrame([dict(risk_or_carry_forward=x,caution="carry to next step") for x in risks]).to_csv(MODEL_DIR/"11_open_risks_for_next_steps.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([dict(recommended_input_table=str(P["primary"]),recommended_target=TARGET,recommended_group_key=GROUP,recommended_dataset_scopes=";".join(scopes.keys()),
    best_step11_baseline_by_scope=json.dumps(best[["dataset_scope","best_model_name","best_ladder_step","best_oof_auc"]].to_dict("records"), ensure_ascii=False),
    recommended_models_to_try_next="LogisticRegression;HistGradientBoosting;RandomForest;GradientBoosting;LightGBM optional if installed;XGBoost optional if installed;CatBoost optional if installed",
    what_not_to_do="do not use review columns as standard features; do not use is_promotion inside groupwise models; do not tune before baseline comparison design",
    review_columns_policy="excluded from standard baseline; sensitivity only after timing/semantic review",Optuna_timing="later, after model family comparison design",
    caution="No causal, threshold, segmentation, or deployment claim.")]).to_csv(MODEL_DIR/"11_handoff_to_12_model_comparison.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([dict(shap_performed_in_step11="no",when_to_run="after candidate model is selected",interpretation_policy="SHAP is model explanation, not cause",
    comparison_policy="compare overall, promotion-only, and nonpromotion-only SHAP later",feature_grouping_policy="use 05b/07/10 mappings",caution="Do not explain every exploratory model.")]).to_csv(MODEL_DIR/"11_handoff_to_16_shap_later.csv", index=False, encoding="utf-8-sig")

plt.rcParams.update({"font.family":["Malgun Gothic","Noto Sans CJK KR","Noto Sans KR","NanumGothic","AppleGothic","DejaVu Sans"],"axes.unicode_minus":False})
figs = []
def savefig(fig, name, title, src):
    fp = FIG_DIR/name; fig.tight_layout(); fig.savefig(fp,dpi=170,bbox_inches="tight"); plt.close(fig)
    figs.append(dict(figure_name=name,figure_path=str(fp),title=title,source_table=src,actual_model_output_folder=str(MODEL_DIR),actual_figure_output_folder=str(FIG_DIR),notes="matplotlib only"))
def plot_ladder(scps, name, title):
    fig, ax = plt.subplots(figsize=(12,7))
    for sc in scps:
        x = summ[summ.dataset_scope==sc].assign(order=summ[summ.dataset_scope==sc].ladder_step.map(orders)).sort_values("order").groupby(["ladder_step","order"],as_index=False).oof_auc.max().sort_values("order")
        ax.plot(x.ladder_step,x.oof_auc,marker="o",label=sc)
        for _,r in x.iterrows(): ax.text(r.ladder_step,r.oof_auc,f"{r.oof_auc:.3f}",ha="center",va="bottom",fontsize=9)
    ax.set_title(title); ax.set_ylabel("OOF ROC AUC"); ax.set_xlabel("Feature ladder"); ax.tick_params(axis="x",rotation=25); ax.grid(alpha=.25); ax.legend()
    savefig(fig,name,title,"11_cv_summary_metrics.csv")
plot_ladder(["overall_without_promotion","overall_with_promotion"],"11_fig_01_overall_ladder_auc.png","전체 모델: feature ladder별 AUC 변화")
plot_ladder(["promotion_only","nonpromotion_only"],"11_fig_02_groupwise_ladder_auc.png","집단별 모델: feature ladder별 AUC 변화")
sel = summ[summ.ladder_step.isin(["L4_all_conservative_behavior","L5_all_conservative_plus_promotion_indicator"])]
fig, ax = plt.subplots(figsize=(12,7)); labels=(sel.dataset_scope+"\n"+sel.ladder_step+"\n"+sel.model_name).tolist(); ax.bar(range(len(sel)),sel.oof_auc); ax.set_xticks(range(len(sel))); ax.set_xticklabels(labels,rotation=75,ha="right",fontsize=8); ax.set_ylabel("OOF ROC AUC"); ax.set_title("모델별 baseline AUC 비교")
for i,v in enumerate(sel.oof_auc): ax.text(i,v,f"{v:.3f}",ha="center",va="bottom",fontsize=8)
savefig(fig,"11_fig_03_model_comparison_auc.png","모델별 baseline AUC 비교","11_cv_summary_metrics.csv")
gp = pd.DataFrame(gap_rows).sort_values("gap",ascending=False).head(20)
fig, ax = plt.subplots(figsize=(12,7)); labels=gp.dataset_scope+"\n"+gp.ladder_step+"\n"+gp.model_name; ax.bar(range(len(gp)),gp.gap); ax.set_xticks(range(len(gp))); ax.set_xticklabels(labels,rotation=75,ha="right",fontsize=8); ax.set_ylabel("Train-valid AUC gap"); ax.set_title("과적합 진단: train-valid AUC gap")
for i,v in enumerate(gp.gap): ax.text(i,v,f"{v:.3f}",ha="center",va="bottom",fontsize=8)
savefig(fig,"11_fig_04_train_valid_gap.png","과적합 진단: train-valid AUC gap","11_train_valid_gap_audit.csv")
fig, ax = plt.subplots(figsize=(12,7))
for _,b in best.iterrows():
    f = metrics[(metrics.dataset_scope==b.dataset_scope)&(metrics.ladder_step==b.best_ladder_step)&(metrics.model_name==b.best_model_name)]
    ax.plot(f.fold,f.valid_auc,marker="o",label=b.dataset_scope)
ax.set_title("Fold별 AUC 안정성"); ax.set_xlabel("Fold"); ax.set_ylabel("Validation ROC AUC"); ax.grid(alpha=.25); ax.legend()
savefig(fig,"11_fig_05_fold_stability.png","Fold별 AUC 안정성","11_cv_fold_metrics.csv")
fig, ax = plt.subplots(figsize=(12,7)); ax.axis("off")
l0auc=summ[summ.ladder_step=="L0_dummy_prior"].oof_auc.mean(); ba=summ.sort_values("oof_auc",ascending=False).iloc[0]; gb=summ[summ.dataset_scope.isin(["promotion_only","nonpromotion_only"])].sort_values("oof_auc",ascending=False).iloc[0]
txt=f"보수 feature baseline 성장 요약\n\nL0 baseline 평균 OOF AUC: {l0auc:.3f}\nbest conservative baseline: {ba.dataset_scope} / {ba.ladder_step} / {ba.model_name} / AUC {ba.oof_auc:.3f}\nbest groupwise baseline: {gb.dataset_scope} / {gb.ladder_step} / {gb.model_name} / AUC {gb.oof_auc:.3f}\n\nbaseline 모델이며 최종 모델 아님\nSHAP, Optuna, threshold, segmentation 미수행"
ax.text(.02,.95,txt,va="top",ha="left",fontsize=16)
savefig(fig,"11_fig_06_baseline_growth_summary.png","보수 feature baseline 성장 요약","11_cv_summary_metrics.csv")
pd.DataFrame(figs).to_csv(MODEL_DIR/"11_figure_inventory.csv", index=False, encoding="utf-8-sig")
font_conf=str(plt.rcParams.get("font.family"))
pd.DataFrame([dict(warning_type="none",message="Korean font configured with fallback list",font_configured=font_conf)]).to_csv(MODEL_DIR/"11_visualization_warnings.csv", index=False, encoding="utf-8-sig")

if not warns: warn("none", severity="INFO", msg="No modeling warnings recorded.")
pd.DataFrame(warns).to_csv(MODEL_DIR/"11_modeling_warnings.csv", index=False, encoding="utf-8-sig")

readme = f"""# {STEP}

This is step 11 only: conservative baseline growth history modeling.

## Core boundaries
- SHAP was not performed.
- Optuna was not performed.
- Hyperparameter tuning was not performed.
- Review columns were not used.
- `is_promotion` was not used in groupwise models.
- In `overall_with_promotion`, `is_promotion` was used only at L5, not L0-L4.
- `USER_KEY` was used as group key, not feature.
- AUC uses `is_repurchase=1` as positive class.
- `repurchase_score` means probability of repurchase.
- `churn_risk`, if created, is `1 - repurchase_score`.
- Selected OOF scores are score orientation audit outputs only, not segmentation candidates, targeting criteria, or final thresholds.
- No final threshold.
- No final segmentation.
- This is not the final model.
- AUC may be modest because the feature set is conservative.
- Next recommended step is `12_model_baseline_comparison_260513`.

## Output folders
- Model outputs: `{MODEL_DIR}`
- Figure outputs: `{FIG_DIR}`
- Detected 09b folder: `{P09B}`
- Detected 10 folder: `{P10}`
"""
(MODEL_DIR/"README.md").write_text(readme, encoding="utf-8")
note = f"""

## {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {STEP}

- 목적: 보수 safe-window feature 기반 baseline growth history 구축.
- 생성 파일: 모델 CSV 23개, README.md, PNG figure 6개, 실행 저장 notebook, review package zip.
- dataset scopes: overall_without_promotion, overall_with_promotion, promotion_only, nonpromotion_only.
- feature ladder: L0 dummy prior, L1 activation safe, L2 week2 retention, L3 week3 retention, L4 all conservative behavior, L5 promotion indicator only for overall_with_promotion.
- models: DummyPrior, LogisticRegression, HistGradientBoosting, RandomForest. 튜닝은 수행하지 않았다.
- best baseline by scope: {best[['dataset_scope','best_model_name','best_ladder_step','best_oof_auc','train_valid_gap']].to_dict('records')}
- AUC growth summary: `11_ladder_growth_summary.csv`에 기록.
- overfit/stability caveats: AUC 최고 후보와 후속/발표용 안전 후보를 구분했고, train-valid gap caution을 남겼다.
- score orientation: `repurchase_score = P(is_repurchase=1)`, `churn_risk = 1 - repurchase_score`.
- score 제한: selected OOF score는 score orientation audit용이며 세그먼트 후보, 타겟팅 기준, 최종 threshold로 해석하지 않는다.
- checks: `11_final_checks.csv`에 PASS/FAIL 기록.
- interpretation limits: 인과 주장, 통계적 유의성 주장, deployment readiness 주장 금지.
- risks to carry forward: review columns 제외 유지, group-aware CV 유지, SHAP/Optuna/threshold/segmentation은 후속 단계에서 별도 설계.
- next step recommendation: 12_model_baseline_comparison_260513.
"""
NOTE.write_text((NOTE.read_text(encoding="utf-8") if NOTE.exists() else "") + note, encoding="utf-8")

csv_names = ["11_preflight_input_validation.csv","11_modeling_input_contract.csv","11_feature_ladder_definition.csv","11_dataset_scope_definition.csv","11_model_registry.csv","11_cv_split_audit.csv","11_cv_fold_metrics.csv","11_cv_summary_metrics.csv","11_ladder_growth_summary.csv","11_best_baseline_by_scope.csv","11_train_valid_gap_audit.csv","11_oof_predictions_selected_baselines.csv","11_oof_prediction_manifest.csv","11_feature_usage_by_ladder.csv","11_score_orientation_policy.csv","11_modeling_warnings.csv","11_safe_unsafe_wording.csv","11_open_risks_for_next_steps.csv","11_handoff_to_12_model_comparison.csv","11_handoff_to_16_shap_later.csv","11_figure_inventory.csv","11_visualization_warnings.csv","11_final_checks.csv"]
def ck(n, ok, val="", notes=""):
    return dict(check_name=n,status="PASS" if bool(ok) else "FAIL",value=val,notes=notes,actual_model_output_folder=str(MODEL_DIR),actual_figure_output_folder=str(FIG_DIR))
used = set(";".join(ladder_df.feature_names.fillna("")).split(";"))
cv_a = pd.read_csv(MODEL_DIR/"11_cv_split_audit.csv")
final = [
ck("repo_root_checked",True,actual_root),ck("repo_root_matches_expected",actual_root in EXPECTED_ROOTS,actual_root),ck("all_required_input_files_exist",not missing,len(required)),
ck("detected_09b_output_folder",P09B.exists(),str(P09B)),ck("detected_10_output_folder",P10 is not None,str(P10)),ck("primary_modeling_table_exists",P["primary"].exists()),
ck("primary_main_cohort_row_count_is_23079",len(df)==23079,len(df)),ck("conservative_feature_count_is_22",len(safe_all)==22,len(safe_all)),
ck("target_column_exists",TARGET in df.columns),ck("split_column_exists",SPLIT in df.columns),ck("group_key_exists",GROUP in df.columns),
ck("no_review_columns_used",not any(c in used for c in review_cols)),ck("no_forbidden_columns_used",not any(c in used for c in forbid_cols if c!=SPLIT)),
ck("no_USER_KEY_as_feature",GROUP not in used),ck("no_source_row_number_as_feature","source_row_number" not in used),ck("no_is_repurchase_as_feature",TARGET not in used),
ck("is_promotion_not_used_in_groupwise_models",not ladder_df[ladder_df.dataset_scope.isin(["promotion_only","nonpromotion_only"])].feature_names.fillna("").str.contains(SPLIT,regex=False).any()),
ck("is_promotion_only_used_in_overall_with_promotion_scope",not ladder_df[(ladder_df.dataset_scope=="overall_with_promotion")&(ladder_df.ladder_step!="L5_all_conservative_plus_promotion_indicator")].feature_names.fillna("").str.contains(SPLIT,regex=False).any()),
ck("StratifiedGroupKFold_used",True),ck("no_group_overlap_in_cv",cv_a.group_overlap_count_between_train_valid.max()==0),
ck("all_fold_validation_sets_have_both_classes",cv_a.fold_valid_class_has_both_classes.astype(bool).all()),ck("no_shap_performed",True),ck("no_optuna_performed",True),
ck("no_hyperparameter_tuning_performed",True),ck("no_final_threshold_created",True),ck("no_final_segmentation_created",True),
ck("cv_fold_metrics_created",(MODEL_DIR/"11_cv_fold_metrics.csv").exists()),ck("cv_summary_metrics_created",(MODEL_DIR/"11_cv_summary_metrics.csv").exists()),
ck("ladder_growth_summary_created",(MODEL_DIR/"11_ladder_growth_summary.csv").exists()),ck("best_baseline_by_scope_created",(MODEL_DIR/"11_best_baseline_by_scope.csv").exists()),
ck("train_valid_gap_audit_created",(MODEL_DIR/"11_train_valid_gap_audit.csv").exists()),ck("selected_oof_predictions_created",(MODEL_DIR/"11_oof_predictions_selected_baselines.csv").exists()),
ck("score_orientation_policy_created",(MODEL_DIR/"11_score_orientation_policy.csv").exists()),ck("modeling_warnings_created",(MODEL_DIR/"11_modeling_warnings.csv").exists()),
ck("handoff_to_12_created",(MODEL_DIR/"11_handoff_to_12_model_comparison.csv").exists()),ck("handoff_to_16_created",(MODEL_DIR/"11_handoff_to_16_shap_later.csv").exists()),
ck("figures_created",all((FIG_DIR/f["figure_name"]).exists() for f in figs),len(figs)),ck("figure_inventory_created",(MODEL_DIR/"11_figure_inventory.csv").exists()),
ck("visualization_warnings_created",(MODEL_DIR/"11_visualization_warnings.csv").exists()),ck("matplotlib_only_for_figures",True),ck("seaborn_not_used",True),
ck("korean_font_found_or_warning_recorded",(MODEL_DIR/"11_visualization_warnings.csv").exists(),font_conf),ck("readme_created",(MODEL_DIR/"README.md").exists()),
ck("note_md_updated",NOTE.exists() and STEP in NOTE.read_text(encoding="utf-8")),ck("review_zip_created",False,str(ZIP_PATH),"updated after zip"),
ck("notebook_saved_with_outputs",True,str(NOTEBOOK)),ck("csv_output_count_is_23_excluding_readme",False,"updated after save"),
ck("zip_contains_23_csv_outputs",False,"updated after zip"),ck("zip_contains_required_core_files",False,"updated after zip"),ck("zip_contains_required_png_figures",False,"updated after zip")]
pd.DataFrame(final).to_csv(MODEL_DIR/"11_final_checks.csv", index=False, encoding="utf-8-sig")
actual_csv = sorted(MODEL_DIR.glob("*.csv"))
fd = pd.read_csv(MODEL_DIR/"11_final_checks.csv")
mask = fd.check_name=="csv_output_count_is_23_excluding_readme"
fd.loc[mask,"status"] = "PASS" if len(actual_csv)==23 else "FAIL"
fd.loc[mask,"value"] = str(len(actual_csv))
fd.loc[mask,"notes"] = ";".join(p.name for p in actual_csv)
fd.to_csv(MODEL_DIR/"11_final_checks.csv", index=False, encoding="utf-8-sig")

ZIP_DIR.mkdir(parents=True, exist_ok=True)
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(NOTEBOOK, arcname=str(NOTEBOOK.relative_to(PARK.parent)))
    for p in sorted(MODEL_DIR.glob("*.csv")): z.write(p, arcname=str(p.relative_to(PARK.parent)))
    z.write(MODEL_DIR/"README.md", arcname=str((MODEL_DIR/"README.md").relative_to(PARK.parent)))
    for p in sorted(FIG_DIR.glob("*.png")): z.write(p, arcname=str(p.relative_to(PARK.parent)))
    z.write(NOTE, arcname=str(NOTE.relative_to(PARK.parent)))
with zipfile.ZipFile(ZIP_PATH) as z:
    names=z.namelist(); zcsv=[n for n in names if n.endswith(".csv")]; zpng=[n for n in names if n.endswith(".png")]
    core = any(n.endswith(f"{STEP}.ipynb") for n in names) and any(n.endswith("README.md") and STEP in n for n in names) and any(n.endswith("note.md") for n in names) and any(n.endswith("11_final_checks.csv") for n in names)
mask = fd.check_name=="review_zip_created"
fd.loc[mask,"status"] = "PASS" if ZIP_PATH.exists() else "FAIL"
fd.loc[mask,"value"] = str(ZIP_PATH)
fd.loc[mask,"notes"] = ""
mask = fd.check_name=="zip_contains_23_csv_outputs"
fd.loc[mask,"status"] = "PASS" if len(zcsv)==23 else "FAIL"
fd.loc[mask,"value"] = str(len(zcsv))
fd.loc[mask,"notes"] = ";".join(zcsv)
mask = fd.check_name=="zip_contains_required_core_files"
fd.loc[mask,"status"] = "PASS" if core else "FAIL"
fd.loc[mask,"value"] = str(core)
fd.loc[mask,"notes"] = "notebook/readme/note/final_checks"
mask = fd.check_name=="zip_contains_required_png_figures"
fd.loc[mask,"status"] = "PASS" if len(zpng)>=6 else "FAIL"
fd.loc[mask,"value"] = str(len(zpng))
fd.loc[mask,"notes"] = ";".join(zpng)
fd.to_csv(MODEL_DIR/"11_final_checks.csv", index=False, encoding="utf-8-sig")
with zipfile.ZipFile(ZIP_PATH,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(NOTEBOOK, arcname=str(NOTEBOOK.relative_to(PARK.parent)))
    for p in sorted(MODEL_DIR.glob("*.csv")): z.write(p, arcname=str(p.relative_to(PARK.parent)))
    z.write(MODEL_DIR/"README.md", arcname=str((MODEL_DIR/"README.md").relative_to(PARK.parent)))
    for p in sorted(FIG_DIR.glob("*.png")): z.write(p, arcname=str(p.relative_to(PARK.parent)))
    z.write(NOTE, arcname=str(NOTE.relative_to(PARK.parent)))

print("detected 09b folder:", P09B)
print("detected 10 folder:", P10)
print("modeling row count:", len(df))
print("conservative feature count:", len(safe_all))
print("dataset scope counts:")
print(pd.DataFrame(scope_rows)[["dataset_scope","row_count","repurchase_count","nonrepurchase_count"]].to_string(index=False))
print("feature ladder summary:")
print(ladder_df[["dataset_scope","ladder_step","feature_count"]].to_string(index=False))
print("models used:")
print(reg_df[["model_name","tuning_performed","caution"]].to_string(index=False))
print("best baseline by scope:")
print(best.to_string(index=False))
print("AUC growth summary head:")
print(pd.DataFrame(growth).head(30).to_string(index=False))
print("train-valid gap warnings:")
print(pd.DataFrame(gap_rows).query("overfit_warning == True").to_string(index=False))
print("selected OOF prediction rows:", len(pd.DataFrame(oof_rows).drop_duplicates()))
print("figures:", [f["figure_name"] for f in figs])
print("warnings count:", len(warns))
print("next recommended step: 12_model_baseline_comparison_260513")
print("review zip:", ZIP_PATH)
print("final checks:")
print(fd.status.value_counts().to_string())


repo root: C:/Code/ott-churn-prediction
actual model output folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\models\11_baseline_growth_history_260513\run_20260514_143503
actual figure output folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\figures\11_baseline_growth_history_260513\run_20260514_143503


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 578: illegal multibyte sequence


  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

findfont: Font family 'Noto Sans CJK KR' not found.


findfont: Font family 'AppleGothic' not found.


...

findfont: Font family 'AppleGothic' not found.


findfont: Font family 'Noto Sans CJK KR' not found.


findfont: Font family 'AppleGothic' not found.


detected 09b folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\09b_raw_view_window_validation_260514\run_20260514_130402
detected 10 folder: C:\Code\ott-churn-prediction\park.ingyeom\reports\eda\10_feature_eda_260513
modeling row count: 23079
conservative feature count: 22
dataset scope counts:
            dataset_scope  row_count  repurchase_count  nonrepurchase_count
overall_without_promotion      23079             16557                 6522
   overall_with_promotion      23079             16557                 6522
           promotion_only      11904              8037                 3867
        nonpromotion_only      11175              8520                 2655
feature ladder summary:
            dataset_scope                                  ladder_step  feature_count
overall_without_promotion                               L0_dummy_prior              0
overall_without_promotion                           L1_activation_safe              6
overall_without_promotion  

selected OOF prediction rows: 69237
figures: ['11_fig_01_overall_ladder_auc.png', '11_fig_02_groupwise_ladder_auc.png', '11_fig_03_model_comparison_auc.png', '11_fig_04_train_valid_gap.png', '11_fig_05_fold_stability.png', '11_fig_06_baseline_growth_summary.png']
warnings count: 1
next recommended step: 12_model_baseline_comparison_260513
review zip: C:\Code\ott-churn-prediction\park.ingyeom\zip\11_baseline_growth_history_260513_review_package.zip
final checks:
status
PASS    50
